# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading, exploring, and analyzing the FAIR² clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

> **Citation**: Liu, Y, Duan, X, Yang, S, Zhang, Y and Han, S 2026. Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution. Frontiers.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset (Croissant schema) URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

# Display dataset title and description
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets and fields. All entities will be referenced by their `@id` fields.

Let's view the list of record sets (tables) present in the dataset, along with their `@id` and their fields.

In [ ]:
# List available record sets
record_sets = metadata_obj.record_sets
if not record_sets:
    print("No RecordSets found in metadata. This may occur if the dataset reference is indirect.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  description: {getattr(rs, 'description', '')}")
        if hasattr(rs, 'fields'):
            print(f"  Fields:")
            for field in rs.fields:
                print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        print()


If you know a RecordSet `@id` (e.g., `cr:RecordSet/ClinicopathologicalTable`), you can quickly preview its records as follows (replace `<record_set_id>`):

In [ ]:
# Example: iterate over records for a given RecordSet by @id
# Replace <record_set_id> below with your chosen RecordSet @id, e.g. 'cr:RecordSet/ClinicopathologicalTable'
example_record_set_id = None
for rs in dataset.metadata.record_sets:
    example_record_set_id = rs.id
    break

if example_record_set_id is not None:
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i > 2:
            break
else:
    print("No RecordSet found in dataset.")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis, referencing each RecordSet by its `@id`.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames
dataframes = dict()

if hasattr(metadata_obj, 'record_sets') and metadata_obj.record_sets:
    record_set_ids = [rs.id for rs in metadata_obj.record_sets]
    for rs_id in record_set_ids:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        dataframes[rs_id] = df

    print(f"Loaded DataFrames for RecordSet @ids: {record_set_ids}")

    # Preview columns from the first (main) RecordSet
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in RecordSet {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No RecordSet in the metadata; cannot extract tabular data.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. All fields are referenced by their `@id`.

**Example steps:**
- Filter records where a numeric field (e.g., `Age` or similar) exceeds a given threshold
- Normalize the numeric field
- Group data by a categorical field (e.g., `Sex` or `Anatomical location`)


In [ ]:
# --- Adapt field IDs below to match your schema ---

# Choose the main RecordSet for analysis
record_set_id = None
for rs in dataset.metadata.record_sets:
    record_set_id = rs.id
    break

df = dataframes.get(record_set_id, None)
if df is None:
    raise ValueError("Could not find DataFrame for selected RecordSet.")

# List possible numeric fields by checking columns
print("Available columns:", df.columns.tolist())

# Pick a numeric field by its @id (column name)
# For demonstration, let's try known names: e.g. 'Age', otherwise use the first numeric column
numeric_field = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field = col
        break

if numeric_field is None:
    # Fallback: pick the first integer/float-like column
    try:
        numeric_cols = df.select_dtypes(include=['float', 'int']).columns
        if len(numeric_cols):
            numeric_field = numeric_cols[0]
    except:
        numeric_field = df.columns[0]  # As last resort

print(f"Using numeric field: {numeric_field}")

# Choose a group (categorical) field, e.g. 'Sex' or 'Anatomical location'
group_field = None
for col in df.columns:
    if any(s in col.lower() for s in ["sex", "gender", "location", "site"]):
        group_field = col
        break

print(f"Using group-by field: {group_field}")

# Filtering: Only numeric values greater than a threshold (set threshold to median if unsure)
if numeric_field is not None:
    try:
        threshold = df[numeric_field].dropna().astype(float).median()
    except Exception:
        threshold = 0
    print(f"Using filtering threshold: {threshold}")
    # Ensure field is numeric-like for filtering
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (showing up to 5):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by the chosen categorical field (if present)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} by {group_field} in filtered records:")
        print(grouped_df)
else:
    print("No numeric field could be determined for analysis.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib or seaborn. Example: plot the filtered and normalized numeric field, with respect to the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric value distribution after filtering and normalization
if numeric_field and (filtered_df is not None):
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field} (> {threshold})')
    plt.xlabel(numeric_field)
    plt.show()

    # If group_field available, plot group means
    if group_field in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field, y=numeric_field, data=filtered_df, ci=None)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data or no numeric field for plotting.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library. We referenced all data structures by their `@id`s, extracted tabular data, performed numeric analysis and grouping, and visualized the results.

**Key Takeaways:**
- The dataset provides rich clinical and molecular information about second primary colorectal cancer in cancer survivors
- Data exploration can reveal trends in key variables, helping to support further research or model building on clinicopathological predictors.

_Always refer to the original Croissant schema for detailed entity and field `@id` references._